# 102 — TCGA RNA-seq Cohort Freeze

## Objective

This notebook freezes the TCGA primary tumor RNA-seq STAR Counts cohort used as the tumor-level discovery input for this project.

The purpose of this notebook is not to preprocess expression values or perform biological analysis. Its purpose is to define, validate, and version-control the exact file-level TCGA/GDC cohort generated from a GDC Data Portal query.

## GDC query

The canonical version-controlled cohort-definition inputs were exported from the GDC Data Portal using the following file-level filters:

- Program: TCGA
- Data Category: Transcriptome Profiling
- Data Type: Gene Expression Quantification
- Experimental Strategy: RNA-Seq
- Workflow Type: STAR - Counts
- Access: Open
- Tissue Type: Tumor
- Tumor Descriptor: Primary

The same query metadata are also recorded in `config/raw_data_registry.json`, which is used below as the structured source of dataset provenance.

## Cohort-freeze rationale

GDC portal queries are dynamic, and portal-level counts may reflect query-context summaries rather than the final file-linked cohort used by this repository. For this reason, the reproducible cohort definition is based on the canonical version-controlled GDC manifest and sample sheet, not on portal counts alone.

At query/export time, the GDC portal reported 10,308 matching STAR Counts files and displayed 10,328 associated TCGA cases. After freezing the downloaded file-level manifest and sample sheet, the resulting cohort contains:

- 10,308 RNA-seq STAR Counts files
- 10,174 unique samples
- 10,122 unique cases
- 33 TCGA projects

Therefore, downstream analyses should use the manifest-derived file index generated by this notebook as the operational cohort definition.

## Outputs

This notebook reads the following canonical version-controlled cohort-definition inputs:

- `config/manifests/tcga_rna/gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt`
- `config/manifests/tcga_rna/gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv`
- `config/manifests/tcga_rna/gdc_metadata_tcga_primary_tumor_rnaseq_star_counts.json`

It validates those inputs and writes the following derived cohort-freeze artifacts:

- `config/manifests/tcga_rna/gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv`
- `config/manifests/tcga_rna/gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json`

## Scope

At this stage, the cohort is frozen at the file level. Any later reduction to one sample per case, project-level filtering, expression matrix construction, or tumor-program analysis will be handled explicitly in downstream TCGA preprocessing notebooks.

This TCGA cohort is intended as a tumor-level discovery layer for recurrent transcriptomic program analysis. It is not intended as a clinical resistance predictor, causal inference layer, or therapeutic validation dataset.

---

In [1]:
# =============================================================================
# 102 — TCGA RNA-seq Cohort Freeze
# Imports
# =============================================================================

import json
import pandas as pd

In [2]:
# =============================================================================
# Project utilities
# =============================================================================

from pancancer_epigenetics.utils.paths import Paths
from pancancer_epigenetics.utils.file_checks import (
    calculate_sha256,
    get_file_size_mb,
    to_project_relative_posix_path,
)
from pancancer_epigenetics.utils.raw_data_registry import (
    get_cohort,
    get_dataset,
    load_raw_data_registry,
    resolve_canonical_dir,
    resolve_canonical_file_path,
)


In [3]:
# =============================================================================
# Load selected TCGA source registry cohort
# =============================================================================

dataset_id = "tcga"
cohort_id = "rnaseq_star_counts"

raw_data_registry = load_raw_data_registry()
tcga_registry = get_dataset(raw_data_registry, dataset_id)
tcga_cohort = get_cohort(raw_data_registry, dataset_id, cohort_id)

required_tcga_registry_fields = {
    "source_database",
    "provider",
    "download_page_url",
    "retrieval_method",
    "cohorts",
}
required_tcga_cohort_fields = {
    "release",
    "canonical_dir",
    "filters",
    "files",
}

missing_tcga_registry_fields = required_tcga_registry_fields - set(tcga_registry)
missing_tcga_cohort_fields = required_tcga_cohort_fields - set(tcga_cohort)

if missing_tcga_registry_fields:
    raise KeyError(
        "Missing required TCGA registry fields: "
        + ", ".join(sorted(missing_tcga_registry_fields))
    )

if missing_tcga_cohort_fields:
    raise KeyError(
        "Missing required TCGA cohort fields: "
        + ", ".join(sorted(missing_tcga_cohort_fields))
    )

tcga_query = tcga_cohort["filters"]

print(f"TCGA registry cohort loaded: {dataset_id}.{cohort_id}")


TCGA registry cohort loaded: tcga.rnaseq_star_counts


In [4]:
# =============================================================================
# Define canonical GDC cohort-definition input paths from registry roles
# =============================================================================

CANONICAL_TCGA_MANIFEST_DIR = resolve_canonical_dir(tcga_cohort)

CANONICAL_MANIFEST_PATH = resolve_canonical_file_path(
    raw_data_registry,
    dataset_id,
    cohort_id,
    "manifest",
)
CANONICAL_SAMPLE_SHEET_PATH = resolve_canonical_file_path(
    raw_data_registry,
    dataset_id,
    cohort_id,
    "sample_sheet",
)
CANONICAL_METADATA_PATH = resolve_canonical_file_path(
    raw_data_registry,
    dataset_id,
    cohort_id,
    "metadata_json",
)

print("Required TCGA registry file roles validated.")


Required TCGA registry file roles validated.


In [5]:
# =============================================================================
# Derived cohort-freeze output paths
# =============================================================================

DERIVED_COHORT_DIR = Paths.config / "manifests" / "tcga_rna"
DERIVED_COHORT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE_INDEX_PATH = DERIVED_COHORT_DIR / "gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv"
OUT_COHORT_FREEZE_PATH = DERIVED_COHORT_DIR / "gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json"


In [6]:
# =============================================================================
# Validate canonical input availability
# =============================================================================

canonical_input_paths = {
    "manifest": CANONICAL_MANIFEST_PATH,
    "sample_sheet": CANONICAL_SAMPLE_SHEET_PATH,
    "metadata_json": CANONICAL_METADATA_PATH,
}

missing_files = [path for path in canonical_input_paths.values() if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Missing required canonical TCGA cohort-definition files:\n"
        + "\n".join(str(path) for path in missing_files)
    )

for label, path in canonical_input_paths.items():
    print(f"{label}: {get_file_size_mb(path):.2f} MB")

manifest: 1.60 MB
sample_sheet: 2.36 MB
metadata_json: 25.82 MB


In [7]:
# =============================================================================
# Load GDC manifest and sample sheet
# =============================================================================

manifest_df = pd.read_csv(CANONICAL_MANIFEST_PATH, sep="\t")
sample_sheet_df = pd.read_csv(CANONICAL_SAMPLE_SHEET_PATH, sep="\t")

print("Manifest shape:", manifest_df.shape)
print("Sample sheet shape:", sample_sheet_df.shape)

display(manifest_df.head())
display(sample_sheet_df.head())

Manifest shape: (10308, 5)
Sample sheet shape: (10308, 11)


,id,filename,md5,size,state
0,744a6d3d-b666-49aa-8d26-47f34e3d1eb5,94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.a...,4a91710ffccd29966a3374ffaf14abec,4231952,released
1,4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5,df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.a...,af51e54b9dc2f6aecd39ab26d2b79824,4238832,released
2,1ace2a0c-773d-45b5-8fd6-968c88731bbb,c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.a...,ff767304460d3385c372d7e0818cb45a,4239626,released
3,7db46624-6d07-4f49-b0ff-18bb41375de9,b1f7af2a-a39b-44c7-a077-e1df6b18268a.rna_seq.a...,82efe74a8659e27431b3a6c2970b9dcb,4236752,released
4,25e923e1-24ce-44a9-934c-20df73d33686,acfb6bc0-2972-48f8-9926-5a5ef46e8d86.rna_seq.a...,7d9634f48654e3a41c5d1e192a6180e6,4229913,released


,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Tissue Type,Tumor Descriptor,Specimen Type,Preservation Method
0,744a6d3d-b666-49aa-8d26-47f34e3d1eb5,94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-BH-A18H,TCGA-BH-A18H-01A,Tumor,Primary,Solid Tissue,OCT
1,4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5,df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-E2-A14P,TCGA-E2-A14P-01A,Tumor,Primary,Solid Tissue,Unknown
2,1ace2a0c-773d-45b5-8fd6-968c88731bbb,c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-AN-A04A,TCGA-AN-A04A-01A,Tumor,Primary,Solid Tissue,OCT
3,7db46624-6d07-4f49-b0ff-18bb41375de9,b1f7af2a-a39b-44c7-a077-e1df6b18268a.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-KIRP,TCGA-MH-A855,TCGA-MH-A855-01A,Tumor,Primary,Solid Tissue,OCT
4,25e923e1-24ce-44a9-934c-20df73d33686,acfb6bc0-2972-48f8-9926-5a5ef46e8d86.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-KIRP,TCGA-P4-A5E6,TCGA-P4-A5E6-01A,Tumor,Primary,Solid Tissue,OCT


In [8]:
# =============================================================================
# Validate manifest and sample sheet structure
# =============================================================================

expected_manifest_columns = {"id", "filename", "md5", "size", "state"}
expected_sample_sheet_columns = {
    "File ID",
    "File Name",
    "Data Category",
    "Data Type",
    "Project ID",
    "Case ID",
    "Sample ID",
    "Tissue Type",
    "Tumor Descriptor",
    "Specimen Type",
    "Preservation Method",
}

missing_manifest_columns = expected_manifest_columns - set(manifest_df.columns)
missing_sample_sheet_columns = expected_sample_sheet_columns - set(sample_sheet_df.columns)

if missing_manifest_columns:
    raise ValueError(f"Missing manifest columns: {sorted(missing_manifest_columns)}")

if missing_sample_sheet_columns:
    raise ValueError(
        f"Missing sample sheet columns: {sorted(missing_sample_sheet_columns)}"
    )

print("Manifest columns validated.")
print("Sample sheet columns validated.")

Manifest columns validated.
Sample sheet columns validated.


In [9]:
# =============================================================================
# Validate manifest and sample sheet consistency
# =============================================================================

manifest_file_ids = set(manifest_df["id"])
sample_sheet_file_ids = set(sample_sheet_df["File ID"])

only_in_manifest = manifest_file_ids - sample_sheet_file_ids
only_in_sample_sheet = sample_sheet_file_ids - manifest_file_ids

print(f"Files in manifest: {len(manifest_file_ids):,}")
print(f"Files in sample sheet: {len(sample_sheet_file_ids):,}")
print(f"File IDs only in manifest: {len(only_in_manifest):,}")
print(f"File IDs only in sample sheet: {len(only_in_sample_sheet):,}")

if only_in_manifest or only_in_sample_sheet:
    raise ValueError("Manifest and sample sheet do not contain the same File IDs.")

file_name_check_df = manifest_df[["id", "filename"]].merge(
    sample_sheet_df[["File ID", "File Name"]],
    left_on="id",
    right_on="File ID",
    how="inner",
    validate="one_to_one",
)

n_filename_mismatches = (
    file_name_check_df["filename"] != file_name_check_df["File Name"]
).sum()

print(f"Filename mismatches: {n_filename_mismatches:,}")

if n_filename_mismatches:
    raise ValueError("Manifest and sample sheet contain mismatched filenames.")

print("Manifest and sample sheet define the same file-level cohort.")

Files in manifest: 10,308
Files in sample sheet: 10,308
File IDs only in manifest: 0
File IDs only in sample sheet: 0
Filename mismatches: 0
Manifest and sample sheet define the same file-level cohort.


In [10]:
# =============================================================================
# Validate frozen cohort definition
# =============================================================================

expected_filename_suffix = ".rna_seq.augmented_star_gene_counts.tsv"

checks = {
    "all_star_counts": manifest_df["filename"].str.endswith(
        expected_filename_suffix, na=False
    ).all(),
    "all_released": (manifest_df["state"] == "released").all(),
    "all_transcriptome_profiling": (
        sample_sheet_df["Data Category"] == "Transcriptome Profiling"
    ).all(),
    "all_gene_expression_quantification": (
        sample_sheet_df["Data Type"] == "Gene Expression Quantification"
    ).all(),
    "all_tumor": (sample_sheet_df["Tissue Type"] == "Tumor").all(),
    "all_primary": (sample_sheet_df["Tumor Descriptor"] == "Primary").all(),
}

for check_name, passed in checks.items():
    print(f"{check_name}: {passed}")

failed_checks = [name for name, passed in checks.items() if not passed]

if failed_checks:
    raise ValueError(
        "Frozen TCGA cohort failed expected definition checks: "
        + ", ".join(failed_checks)
    )

print("Frozen cohort definition validated.")

all_star_counts: True
all_released: True
all_transcriptome_profiling: True
all_gene_expression_quantification: True
all_tumor: True
all_primary: True
Frozen cohort definition validated.


In [11]:
# =============================================================================
# Build frozen TCGA RNA-seq file index
# =============================================================================

manifest_index_df = manifest_df.rename(
    columns={
        "id": "file_id",
        "filename": "file_name",
        "md5": "md5",
        "size": "file_size_bytes",
        "state": "state",
    }
)

sample_sheet_index_df = sample_sheet_df.rename(
    columns={
        "File ID": "file_id",
        "File Name": "file_name_sample_sheet",
        "Data Category": "data_category",
        "Data Type": "data_type",
        "Project ID": "project_id",
        "Case ID": "case_id",
        "Sample ID": "sample_id",
        "Tissue Type": "tissue_type",
        "Tumor Descriptor": "tumor_descriptor",
        "Specimen Type": "specimen_type",
        "Preservation Method": "preservation_method",
    }
)

file_index_df = manifest_index_df.merge(
    sample_sheet_index_df,
    on="file_id",
    how="inner",
    validate="one_to_one",
)

file_name_mismatches = (
    file_index_df["file_name"] != file_index_df["file_name_sample_sheet"]
).sum()

if file_name_mismatches:
    raise ValueError("Manifest and sample sheet filenames do not match.")

file_index_df = file_index_df.drop(columns=["file_name_sample_sheet"])

file_index_df = file_index_df[
    [
        "file_id",
        "file_name",
        "md5",
        "file_size_bytes",
        "state",
        "project_id",
        "case_id",
        "sample_id",
        "data_category",
        "data_type",
        "tissue_type",
        "tumor_descriptor",
        "specimen_type",
        "preservation_method",
    ]
].sort_values(["project_id", "case_id", "sample_id", "file_id"])

print("File index shape:", file_index_df.shape)
print("Unique projects:", file_index_df["project_id"].nunique())
print("Unique cases:", file_index_df["case_id"].nunique())
print("Unique samples:", file_index_df["sample_id"].nunique())

display(file_index_df.head())

File index shape: (10308, 14)
Unique projects: 33
Unique cases: 10122
Unique samples: 10174


,file_id,file_name,md5,file_size_bytes,state,project_id,case_id,sample_id,data_category,data_type,tissue_type,tumor_descriptor,specimen_type,preservation_method
2133,fe16b2d3-17b0-4e24-ab31-62d2e951b3a2,6bacf042-830f-47dd-bac3-5696aebf2574.rna_seq.a...,967a3efd89a8d709df7170d3be032580,4225986,released,TCGA-ACC,TCGA-OR-A5J1,TCGA-OR-A5J1-01A,Transcriptome Profiling,Gene Expression Quantification,Tumor,Primary,Unknown,OCT
2142,f11aa62e-5d67-4e49-8f64-5ed663a3b424,06cc0bc6-1b36-4318-99ee-dfc83cc134c0.rna_seq.a...,9f41ec17c77ac638baaf7d7e065f03df,4233682,released,TCGA-ACC,TCGA-OR-A5J2,TCGA-OR-A5J2-01A,Transcriptome Profiling,Gene Expression Quantification,Tumor,Primary,Unknown,OCT
2199,3137ba2b-0d6b-4fcc-8512-894fd6a43ff6,171e4e82-93ed-4be7-8b61-7c1df82bde0f.rna_seq.a...,d1afd241c711ac71989ea60c1874a536,4222749,released,TCGA-ACC,TCGA-OR-A5J3,TCGA-OR-A5J3-01A,Transcriptome Profiling,Gene Expression Quantification,Tumor,Primary,Unknown,OCT
2181,b8abd8a3-1525-4bf5-b083-eddefa3ca19e,32d1a985-ad2c-4046-873d-02416f5a68f3.rna_seq.a...,2b2d56f116f7d3d78f008ee80b73d772,4219218,released,TCGA-ACC,TCGA-OR-A5J5,TCGA-OR-A5J5-01A,Transcriptome Profiling,Gene Expression Quantification,Tumor,Primary,Unknown,OCT
2193,224fa76c-cf8d-48d8-8723-15907a04f2d2,a0169877-dd20-4c7d-826c-bb424199a036.rna_seq.a...,ba6a678a758ed96e14768cb2c50bb3a8,4219826,released,TCGA-ACC,TCGA-OR-A5J6,TCGA-OR-A5J6-01A,Transcriptome Profiling,Gene Expression Quantification,Tumor,Primary,Unknown,OCT


In [12]:
# =============================================================================
# Define cohort-freeze metadata
# =============================================================================

cohort_freeze = {
    "cohort_name": "tcga_primary_tumor_rnaseq_star_counts",
    "source_database": tcga_registry["source_database"],
    "provider": tcga_registry["provider"],
    "release": tcga_cohort["release"],
    "retrieval_method": tcga_registry["retrieval_method"],
    "download_page_url": tcga_registry["download_page_url"],
    "filters": tcga_query,
    "canonical_inputs": {
        "manifest": {
            "path": to_project_relative_posix_path(CANONICAL_MANIFEST_PATH, Paths.root),
            "sha256": calculate_sha256(CANONICAL_MANIFEST_PATH),
        },
        "sample_sheet": {
            "path": to_project_relative_posix_path(CANONICAL_SAMPLE_SHEET_PATH, Paths.root),
            "sha256": calculate_sha256(CANONICAL_SAMPLE_SHEET_PATH),
        },
        "metadata_json": {
            "path": to_project_relative_posix_path(CANONICAL_METADATA_PATH, Paths.root),
            "sha256": calculate_sha256(CANONICAL_METADATA_PATH),
        },
    },
    "cohort_counts": {
        "n_files": int(file_index_df["file_id"].nunique()),
        "n_projects": int(file_index_df["project_id"].nunique()),
        "n_cases": int(file_index_df["case_id"].nunique()),
        "n_samples": int(file_index_df["sample_id"].nunique()),
        "total_file_size_bytes": int(file_index_df["file_size_bytes"].sum()),
        "total_file_size_gib": round(
            file_index_df["file_size_bytes"].sum() / 1024**3, 3
        ),
    },
    "derived_outputs": {
        "file_index": to_project_relative_posix_path(OUT_FILE_INDEX_PATH, Paths.root),
        "cohort_freeze": to_project_relative_posix_path(OUT_COHORT_FREEZE_PATH, Paths.root),
    },
    "reproducibility_note": (
        "This TCGA cohort was generated from a GDC Data Portal query. "
        "Because portal queries are dynamic, the canonical version-controlled manifest "
        "and derived file index define the exact frozen file-level cohort used by this repository."
    ),
    "scientific_scope": (
        "Tumor-level discovery input for recurrent transcriptomic program analysis. "
        "This cohort freeze does not imply clinical resistance prediction, causal inference, "
        "or therapeutic validation."
    ),
}

print(json.dumps(cohort_freeze, indent=2))


{
  "cohort_name": "tcga_primary_tumor_rnaseq_star_counts",
  "source_database": "Genomic Data Commons (GDC)",
  "provider": "National Cancer Institute",
  "release": "GDC Data Portal query export, 2026-06-12",
  "retrieval_method": "GDC Data Portal query-derived cohort export",
  "download_page_url": "https://portal.gdc.cancer.gov/repository",
  "filters": {
    "program": "TCGA",
    "data_category": "Transcriptome Profiling",
    "data_type": "Gene Expression Quantification",
    "experimental_strategy": "RNA-Seq",
    "workflow_type": "STAR - Counts",
    "access": "Open",
    "tissue_type": "Tumor",
    "tumor_descriptor": "Primary"
  },
  "canonical_inputs": {
    "manifest": {
      "path": "config/manifests/tcga_rna/gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt",
      "sha256": "3fcbb6353753d47a851826005500ca6de7f58a3734479075e390bd33e712cc3d"
    },
    "sample_sheet": {
      "path": "config/manifests/tcga_rna/gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.t

In [13]:
# =============================================================================
# Write derived cohort-freeze outputs
# =============================================================================

DERIVED_COHORT_DIR.mkdir(parents=True, exist_ok=True)

file_index_df.to_csv(OUT_FILE_INDEX_PATH, sep="	", index=False)

with OUT_COHORT_FREEZE_PATH.open("w", encoding="utf-8") as handle:
    json.dump(cohort_freeze, handle, indent=2)

print("Wrote derived cohort-freeze outputs:")
print(f"- {to_project_relative_posix_path(OUT_FILE_INDEX_PATH, Paths.root)}")
print(f"- {to_project_relative_posix_path(OUT_COHORT_FREEZE_PATH, Paths.root)}")


Wrote derived cohort-freeze outputs:
- config/manifests/tcga_rna/gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv
- config/manifests/tcga_rna/gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json
